# QA Agent — run in Colab

Runs the autonomous QA agent against [Sauce Demo](https://www.saucedemo.com) entirely in the cloud — no local install needed.

**Before running:** upload `qa-agent.zip` to this Colab session (folder icon on the left → upload) so it sits at `/content/qa-agent.zip`. Then run the cells in order.

In [ ]:
# 1. Install dependencies (Colab has its own Python env, this takes ~1-2 min)
!pip install -q google-genai langgraph langchain-core playwright python-dotenv
!playwright install --with-deps chromium

In [ ]:
# 2. Unzip the project (upload qa-agent.zip first via the Files pane on the left)
import zipfile, os

zip_path = "/content/qa-agent.zip"
assert os.path.exists(zip_path), "Upload qa-agent.zip to /content first (Files pane -> upload)"

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall("/content")

import sys
sys.path.insert(0, "/content/qa-agent")
print("Project extracted to /content/qa-agent")

In [ ]:
# 3. Set your Gemini API key (typed input, not stored in the notebook file)
import os
from getpass import getpass

os.environ["GEMINI_API_KEY"] = getpass("Paste your Gemini API key: ")

In [ ]:
# 4. Run the agent against Sauce Demo
from agent.graph import run_requirement

requirement = "user can add an item to the cart"  # ← change this to try other requirements
url = "https://www.saucedemo.com"
context = (
    "This is Sauce Demo. Log in first if the requirement needs it, using "
    "username 'standard_user' and password 'secret_sauce', selectors "
    "#user-name, #password, #login-button. After login the app lands on "
    "the inventory page. 'Add to cart' buttons use data-test attributes "
    "like [data-test='add-to-cart-sauce-labs-backpack]]. Cart badge is "
    "'.shopping_cart_badge'."
)

final_state = run_requirement(requirement, url, context, headless=True)

print("\n── Steps run ──")
for r in final_state["results"]:
    status = "✅" if r.passed else "❌"
    print(f"{status} {r.action} {r.target} — {r.detail}")

if final_state["bugs"]:
    print("\n── Bugs found ──")
    for bug in final_state["bugs"]:
        print(bug.to_markdown())
else:
    print("\n✅ No bugs found — all steps passed.")

### Try another requirement
Just edit `requirement` in the cell above and re-run cell 4 (no need to redo steps 1-3 unless you restart the runtime).